# Triage Copilot — Walkthrough

Three sample alerts run end-to-end through the full pipeline:

1. `impossible_travel` happy path — clean run with grounded verdict
2. `impossible_travel` with stale-clean threat intel — D14 defense in action
3. `ransomware` P0 with T3 Opus escalation

Runs use the deterministic `SequenceClient` with pre-recorded responses so the notebook is reproducible without an Anthropic API key. To run against the live Anthropic API, see the last cell.

In [ ]:
from datetime import UTC, datetime
import json

from triage.enrichment.base import SourceQuery
from triage.enrichment.fanout import build_default_registry, run_fanout
from triage.llm.client import LLMResponse, SequenceClient
from triage.reasoning.agent import reason
from triage.reasoning.escalation import escalate_to_t3, should_escalate
from triage.schemas.alert import Asset, CanonicalAlertEvent, Observable
from triage.schemas.plan_loader import PlanTemplateRegistry
from triage.schemas.verdict import AIMetadata
from triage.validation.validator import validate_response

registry = PlanTemplateRegistry()
sources = build_default_registry()
print(f'plan templates loaded: {registry.families()}')
print(f'sources registered: {sorted(sources.keys())}')

## 1. Happy path: impossible_travel

In [ ]:
alert = CanonicalAlertEvent(
    tenant_id='tenant_a',
    alert_id='nb_demo_01',
    source_system='okta',
    source_adapter_version='okta_v1',
    rule_id='okta.impossible_travel.v3',
    rule_family='impossible_travel',
    received_at=datetime(2026, 6, 17, 14, 32, 11, tzinfo=UTC),
    detected_at=datetime(2026, 6, 17, 14, 32, 10, tzinfo=UTC),
    severity_hint='P1',
    primary_assets=[Asset(asset_id='u_acct_lead', asset_type='user', tenant_id='tenant_a')],
    observables=[Observable(observable_type='ip', value='198.51.100.42', source_field_path='client.ipAddress')],
    summary='Sign-on from Bulgaria 30s after Portland session',
)
plan = registry.build_plan(alert.rule_family, alert.severity_hint)
print(f'plan tier_preference: {plan.tier_preference}')
print(f'plan required_sources: {plan.required_sources}')
print(f'plan optional_sources: {plan.optional_sources}')

In [ ]:
query = SourceQuery(
    tenant_id=alert.tenant_id,
    alert_id=alert.alert_id,
    entity_id=alert.grouping_entity(),
    ioc=alert.primary_ioc(),
    extra={'rule_family': alert.rule_family},
)
bundle = run_fanout(plan, query, sources)
print(f'retrievals: {len(bundle.retrievals)}')
print(f'enrichments_failed: {bundle.enrichments_failed}')
print(f'spans: {[(s["source_type"], s["outcome"]) for s in bundle.spans]}')

In [ ]:
identity_ref = bundle.by_source('identity_store')[0]
verdict_json = {
    'verdict': 'likely_true_positive',
    'confidence': 0.82,
    'severity': 'P1',
    'severity_rationale': 'Geo anomaly with intact MFA.',
    'summary': 'User u_acct_lead authenticated from Bulgaria 30s after Portland session.',
    'attack_chain': [],
    'observed_facts': [{
        'fact_id': 'f1', 'claim': 'User has MFA enabled.',
        'retrieval_id': identity_ref.retrieval_id,
        'field_path': 'mfa_enabled', 'expected_value': True, 'confidence': 0.95,
    }],
    'inferences': [{'inference_id': 'i1', 'claim': 'MFA evidence reduces credential-stuff probability.', 'supported_by_fact_ids': ['f1'], 'confidence': 0.78}],
    'recommendations': [{'priority': 1, 'action': 'force_password_reset', 'rationale': 'Defense in depth.', 'supported_by_inference_ids': ['i1'], 'blast_radius': 'low', 'reversible': True, 'automatable': False}],
    'blast_radius': {'affected_assets': ['u_acct_lead']},
    'uncertainty': {'missing_enrichments': []},
}
client = SequenceClient([LLMResponse(content=json.dumps(verdict_json), stop_reason='end_turn', tokens_in=2000, tokens_out=600, cost_usd=0.022, model='claude-sonnet-4-6')])
response, augmented, extensions = reason(alert, plan, bundle, client, sources=sources)
outcome = validate_response(response.content, augmented, triage_id='triage_nb_01', tenant_id=alert.tenant_id, alert_id=alert.alert_id, investigation_plan_dump=plan.model_dump(), received_at=alert.received_at, ai_metadata=AIMetadata(route_tier='standard_t2', model_chain=['haiku', 'sonnet'], cost_usd=0.022))
print(f'verdict: {outcome.verdict.verdict}')
print(f'confidence: {outcome.verdict.confidence}')
print(f'recommendations[0]: {outcome.verdict.recommendations[0].action}')
print(f'validation failures: {len(outcome.failures)}')
print(f'plan extensions: {extensions}')

## 2. Stale-clean threat intel (D14 defense)

tenant_a sees the IOC 198.51.100.42 as `reputation: unknown` with cached_at 90 days ago. The reasoning agent MUST NOT treat this as benign signal strong enough for `likely_false_positive`.

In [ ]:
ti_refs = bundle.by_source('threat_intel')
if ti_refs:
    ref = ti_refs[0]
    print(f'provider: {ref.provider}')
    print(f'provider_confidence: {ref.provider_confidence}')
    print(f'cached_at: {ref.cached_at}')
    print(f'reputation in payload: {ref.payload.get("reputation")}')
else:
    print('threat_intel is optional for impossible_travel plan; fan-out did not fetch it on this run.')
    print('Run again with override_plan to include threat_intel to see the stale-clean signature.')

## 3. Ransomware P0 with T3 Opus escalation

Should-escalate criteria: T2 returned low confidence AND severity in {P0, P1} AND family in DEEP_FAMILIES.

In [ ]:
rw_alert = CanonicalAlertEvent(
    tenant_id='tenant_a', alert_id='nb_demo_03', source_system='crowdstrike',
    source_adapter_version='crowdstrike_v1', rule_id='cs.ransomware.v2', rule_family='ransomware',
    received_at=datetime(2026, 6, 17, 14, 32, 11, tzinfo=UTC),
    detected_at=datetime(2026, 6, 17, 14, 32, 10, tzinfo=UTC),
    severity_hint='P0',
    primary_assets=[Asset(asset_id='srv_billing_01', asset_type='service', tenant_id='tenant_a')],
    summary='Mass file rename + entropy spike on billing host',
)
rw_plan = registry.build_plan('ransomware', 'P0')
rw_query = SourceQuery(tenant_id='tenant_a', alert_id=rw_alert.alert_id, entity_id='srv_billing_01', ioc=None, extra={'rule_family': 'ransomware'})
rw_bundle = run_fanout(rw_plan, rw_query, sources)

print(f'T2 returned low confidence (0.4) → should_escalate: {should_escalate("P0", "ransomware", 0.4)}')

t3_client = SequenceClient([
    LLMResponse(content=json.dumps({'verdict': 'confirmed_true_positive'}), stop_reason='end_turn', tokens_in=2200, tokens_out=600, cost_usd=0.05, model='claude-opus-4-7'),
    LLMResponse(content=json.dumps({'verdict': 'confirmed_true_positive'}), stop_reason='end_turn', tokens_in=2200, tokens_out=600, cost_usd=0.05, model='claude-opus-4-7'),
    LLMResponse(content=json.dumps({'verdict': 'likely_true_positive'}), stop_reason='end_turn', tokens_in=2200, tokens_out=600, cost_usd=0.05, model='claude-opus-4-7'),
])
escalation = escalate_to_t3(rw_alert, rw_plan, rw_bundle, t3_client, sample_size=3)
print(f'T3 majority verdict: {escalation.majority_verdict}')
print(f'T3 cost: ${escalation.cost_usd:.3f}')

## 4. Live API capture (one cell; uncomment to run)

Set `ANTHROPIC_API_KEY` in env, uncomment, and run. The response is saved as a fixture under `fixtures/llm_replays/<digest>.json` and replayed by `FixtureReplayClient` on subsequent runs (deterministic + reproducible).

This cell does NOT execute on `uv run pytest` or `uv run eval` — it is the explicit capture mechanism documented in DESIGN.md §4.4.

In [ ]:
# from pathlib import Path
# from triage.llm.client import AnthropicClient, FIXTURE_DIR
# import json
# from triage.reasoning.agent import _build_t2_request
#
# req = _build_t2_request(alert, plan, bundle, [])
# live = AnthropicClient()
# resp = live.complete(req)
# fixture_path = FIXTURE_DIR / f'{req.digest()}.json'
# fixture_path.write_text(json.dumps({
#     'content': resp.content, 'stop_reason': resp.stop_reason,
#     'tool_calls': resp.tool_calls, 'tokens_in': resp.tokens_in,
#     'tokens_out': resp.tokens_out, 'cost_usd': resp.cost_usd, 'model': resp.model,
# }))
# print(f'live captured at {datetime.now(UTC).isoformat()} → {fixture_path}')